In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [14]:
class BatsmanState(TypedDict):

    runs: int
    balls: int
    fours: int
    sixes: int
    strike_rate: float
    balls_per_boundary: float
    boundary_percent: float 
    summary: str

In [20]:
def calculate_strike_rate(state: BatsmanState) -> BatsmanState:
    strike_rate = (state['runs']/state['balls'])*100
    return {'strike_rate': strike_rate} #partial update

def calculate_balls_per_boundary(state: BatsmanState) -> BatsmanState:
    bpb = state['balls']/(state['fours'] + state['sixes'])
    return {'balls_per_boundary': bpb}

def calculate_boundary_percent(state: BatsmanState) -> BatsmanState:
    bp = ((state['fours']*4 + state['sixes']*6)/state['runs'])*100
    return {'boundary_percent': bp}

def summary(state: BatsmanState) -> BatsmanState:
    summary = f"""
Strike Rate - {state['strike_rate']}
Balls per boundary - {state['balls_per_boundary']}
Boundary Percetn - {state['boundary_percent']}
    """
    return {'summary': summary}

In [22]:
graph = StateGraph(BatsmanState)
graph.add_node('calculate_strike_rate', calculate_strike_rate)
graph.add_node('calculate_balls_per_boundary', calculate_balls_per_boundary)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)

graph.add_edge(START, 'calculate_balls_per_boundary')
graph.add_edge(START, 'calculate_strike_rate')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calculate_balls_per_boundary', 'summary')
graph.add_edge('calculate_strike_rate', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)

workflow = graph.compile()

In [25]:
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

workflow.invoke(initial_state)


{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'strike_rate': 200.0,
 'balls_per_boundary': 5.0,
 'boundary_percent': 48.0,
 'summary': '\nStrike Rate - 200.0\nBalls per boundary - 5.0\nBoundary Percetn - 48.0\n    '}